# Exact Numerical Calculation of $n_s$ and $r$

This notebook calculates the **Scalar Spectral Index ($n_s$)** and the **Tensor-to-Scalar Ratio ($r$)** by solving the exact Mukhanov-Sasaki equations. 

Unlike the slow-roll approximation, which assumes parameters like $\epsilon$ and $\eta$ are small and constant, this method involves:
1.  Solving the **exact background dynamics** ($a(t), \phi(t), H(t)$).
2.  Finding the exact moment of **horizon crossing** for the pivot scale $k_*$.
3.  Solving the **Mukhanov-Sasaki mode equations** for $k_*$ and its neighbors to determine the precise power spectrum $\mathcal{P}_{\mathcal{R}}(k)$.

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import json
import datetime
import uuid

# Import modules from the current directory (project root)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from inflation_models import HiggsModel, QuadraticModel
import inf_dyn_background as bg_solver
import inf_dyn_MS_full as ms_solver

The simulation works in dimensionless units scaled by the Planck mass $M_{pl}$ and a time-scaling parameter $S$.  

**Core Parameters:**
- **$M_{pl}$**: Reduced Planck Mass ($2.435 \times 10^{18}$ GeV). In the code, field values are explicitly normalized by this (i.e. $x = \phi/M_{pl}$ implies $M_{pl}=1$ unit).
- **$S$**: A pure dimensionless numerical parameter used to scale **time**. The code time $T$ relates to physical time via $T = S \cdot M_{pl} \cdot t$, meaning $dt = dT / (S \cdot M_{pl})$. By the chain rule, physical time derivatives scale as $\frac{d}{dt} = S \cdot M_{pl} \frac{d}{dT}$. This maps the physical timescales into manageable $\mathcal{O}(1)$ simulation steps $\Delta T$.

**Variable Mapping:**

| Code Variable | Exact Definition (Physical $\leftrightarrow$ Code) | Physical Meaning |
| :--- | :--- | :--- |
| **`x`** | $\phi / M_{pl}$ | Dimensionless Field Value |
| **`y`** | $dx/dT = \frac{\dot{\phi}}{S M_{pl}^2}$ | Dimensionless Field Velocity |
| **`z`** | $H / (S M_{pl})$ | Dimensionless Hubble Parameter |
| **`A`** | $a$ | Scale Factor (Dimensionless) |
| **`S`** | $S = 5 \times 10^{-5}$ | **Time Scaling Parameter** (Sets 'tick rate' of simulation) |
| **`M`** | $M = 5.9 \times 10^{-6}$ | **Mass Scale** (Simulated physical mass scale) |
| **`v0`** | $v_0 = 0.5 M^2$ (Quadratic) or $\lambda/(4\xi^2)$ (Higgs) | **Potential Amplitude** (Sets energy scale of inflation) |
| **`v`, `u`** | $\text{Re}(v_k), \text{Im}(v_k)$ | Scalar MS Variable (Dimension: Mass$^{-1/2}$ approx) |
| **`h`, `g`** | $\text{Re}(h_k), \text{Im}(h_k)$ | Tensor MS Variable |
| **`N`** | $\ln a$ | Number of e-folds |

## 2. Background Evolution & Pivot Scale Selection

We simulate the background evolution to find the end of inflation (defined where $\epsilon_H = 1$). We then identify the **pivot scale** $k_*$ as the mode that exits the horizon exactly $N_* = 60$ e-folds before the end.

$$ k_* = a(t_*) H(t_*) \quad \text{at} \quad N(t_*) = N_{\text{end}} - 60 $$

In [15]:
# Setup Model
# Use HiggsModel(xi=1000.0) to match the user's apparent context, or default
model = HiggsModel(xi=17000.0, lam=0.13)  
print(f"Model Initial Ai: {model.Ai}")
print(f"Model S: {model.S}")
print(f"Model xi: {model.xi_val}")

model.phi0= 6.0
model.yi = -0.1

# 1. Run Background
T_span_bg = np.linspace(0, 5000, 10000)
bg_sol = bg_solver.run_background_simulation(model, T_span_bg)
derived_bg = bg_solver.get_derived_quantities(bg_sol, model)

# 2. Find End of Inflation (epsH = 1)
epsH = derived_bg['epsH']
N_efolds = derived_bg['N'] # Number of efolds since start

end_indices = np.where(epsH >= 1)[0]
if len(end_indices) > 0:
    end_idx = end_indices[0]
else:
    end_idx = -1
    print("Warning: Inflation did not end explicitly in the window.")

N_total = N_efolds[end_idx]
print(f"Inflation ends at N_sim = {N_total:.2f} (Index: {end_idx})")

# 3. Identify Pivot Scale (N_star = 60 before end)
N_star = 60.0
N_pivot = N_total - N_star
pivot_idx = np.argmin(np.abs(N_efolds - N_pivot))

# k_code = a * z at pivot
x_bg, y_bg, z_bg, n_bg = bg_sol
a_pivot = np.exp(n_bg[pivot_idx])
z_pivot = z_bg[pivot_idx]
H_pivot_phys = z_pivot * model.S
k_pivot_code = a_pivot * z_pivot
k_pivot_phys = k_pivot_code * model.S

print(f"Pivot scale k* (code units) = {k_pivot_code:.4e}")
print(f"Pivot scale k* (physical units) = {k_pivot_phys:.4e}")
print(f"  -> Exits at Simulation N = {N_efolds[pivot_idx]:.2f}")
print(f"  -> This is {N_total - N_efolds[pivot_idx]:.2f} e-folds before the end of inflation.")
print(f"At pivot: a = {a_pivot:.4e}, z = {z_pivot:.4e}, H_phys = {H_pivot_phys:.4e}")

Model Initial Ai: 1e-05
Model S: 5e-05
Model xi: 17000.0
Inflation ends at N_sim = 78.22 (Index: 1331)
Pivot scale k* (code units) = 1.0180e+02
Pivot scale k* (physical units) = 5.0902e-03
  -> Exits at Simulation N = 18.25
  -> This is 59.98 e-folds before the end of inflation.
At pivot: a = 8.4141e+02, z = 1.2099e-01, H_phys = 6.0496e-06


## 3. Solving the Perturbations

Using the `run_ms_simulation` function from `inf_dyn_MS_full.py`, we evolve the perturbations for a specific $k_{code}$. 

**Initial Conditions**: We use **Bunch-Davies** initial conditions. We start the integration when the mode is deep inside the horizon ($k_{code} \gg a \cdot z$, typically $k \approx 100 aH$).
$$ v_k(\tau_{ini}) = \frac{1}{\sqrt{2k}} e^{-ik\tau} $$

The function returns the power spectra $P_{\mathcal{R}}$ (scalar) and $P_T$ (tensor) at the end of the simulation.

In [16]:
def calculate_for_mode(k_mode_code, model, bg_sol, end_idx):
    # Select start time when k_code ~ 100 a*z (deep inside horizon)
    n_bg = bg_sol[3]
    z_bg = bg_sol[2]
    log_az = n_bg + np.log(z_bg)
    target_start = np.log(k_mode_code) - np.log(100) 
    start_idx = np.argmin(np.abs(log_az[:end_idx] - target_start))
    
    # Ensure we don't start too early if the simulation didn't go back that far
    start_idx = max(start_idx, 0)
    
    # Get ICs from background solution at this time
    xi = bg_sol[0][start_idx]
    yi = bg_sol[1][start_idx]
    zi = bg_sol[2][start_idx]
    Ai = np.exp(bg_sol[3][start_idx])
    
    # Time span: From start_idx to end_idx
    t_start = T_span_bg[start_idx]
    t_end = T_span_bg[end_idx]
    T_ms = np.linspace(t_start, t_end, 5000)
    
    # Run MS using the imported script function
    ms_sol = ms_solver.run_ms_simulation(xi, yi, zi, Ai, T_ms, k_mode_code, model)
    derived = ms_solver.get_ms_derived_quantities(ms_sol, model, k_mode_code)
    
    return derived['P_S'][-1], derived['P_T'][-1] # Return final values

In [20]:
calculate_for_mode(k_code, model, bg_sol, end_idx)

(np.float64(2.4625179217849322e-09), np.float64(7.416198556377861e-12))

## 4. Calculating Observables

We calculate the observables by comparing the power spectrum at the pivot scale $k_*$ and slightly shifted scales $k_{\pm} = k_* (1 \pm \delta)$.

**Spectral Index ($n_s$):**
$$ n_s = 1 + \frac{d \ln \mathcal{P}_{\mathcal{R}}}{d \ln k} \approx 1 + \frac{\ln \mathcal{P}(k_+) - \ln \mathcal{P}(k_-)}{\ln k_+ - \ln k_-} $$

**Tensor-to-Scalar Ratio ($r$):**
$$ r = \frac{\mathcal{P}_T(k_*)}{\mathcal{P}_{\mathcal{R}}(k_*)} $$

In [51]:
# Define k modes
delta = 1e-5
ks_code = [k_pivot_code * (1 - delta), k_pivot_code, k_pivot_code * (1 + delta)]

results = []
print("Calculating power spectra...")
for k_code in ks_code:
    Ps, Pt = calculate_for_mode(k_code, model, bg_sol, end_idx)
    results.append((Ps, Pt))
    print(f"  k_code={k_code:.4e}: Ps={Ps:.4e}")

# Calculate ns
log_k = np.log(ks_code)
log_Ps = np.log([r[0] for r in results])

# Derivative d ln Ps / d ln k
slope = (log_Ps[2] - log_Ps[0]) / (log_k[2] - log_k[0])
ns = 1 + slope

# Calculate r at pivot (middle)
r_val = results[1][1] / results[1][0]

print("-"*30)
print(f"Results for {model.name}:")
print(f"n_s = {ns:.6f}")
print(f"r   = {r_val:.6f}")
print("-"*30)

Calculating power spectra...
  k_code=1.0180e+02: Ps=2.4629e-09
  k_code=1.0180e+02: Ps=2.4629e-09
  k_code=1.0181e+02: Ps=2.4629e-09
------------------------------
Results for Higgs Inflation:
n_s = 0.983996
r   = 0.003011
------------------------------


In [50]:
# Convergence Test for n_s
deltas = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7, 1e-8, 1e-9]
convergence_results = []

print(f"{'delta':<10} | {'n_s':<12}")
print("-" * 40)

for d in deltas:
    # Calculate for two shifted modes
    ks_test = [k_pivot_code * (1 - d), k_pivot_code * (1 + d)]
    Ps_plus, _ = calculate_for_mode(ks_test[1], model, bg_sol, end_idx)
    Ps_minus, _ = calculate_for_mode(ks_test[0], model, bg_sol, end_idx)
    
    # Calculate slope
    slope_test = (np.log(Ps_plus) - np.log(Ps_minus)) / (np.log(ks_test[1]) - np.log(ks_test[0]))
    ns_test = 1 + slope_test
    
    
    print(f"{d:<10.1e} | {ns_test:<12.8f}")
    convergence_results.append(ns_test)

print("-" * 40)


delta      | n_s         
----------------------------------------
1.0e-01    | 0.96746086  
1.0e-02    | 0.97509018  
1.0e-03    | 0.98388010  
1.0e-04    | 0.98398296  
1.0e-05    | 0.98399598  
1.0e-06    | 0.98434015  
1.0e-07    | 0.98452321  
1.0e-08    | 0.99562679  
1.0e-09    | 0.80678213  
----------------------------------------


## 5. Saving Results

We save all relevant simulation data (model parameters, physical constants, initial conditions, and final observables) into a structured JSON file. This ensures reproducibility and ease of analysis.

In [54]:
def save_simulation_results(model, ns, r_val, delta, k_pivot_code, N_total, N_pivot, results_list, ks_code_list, output_dir="../outputs/results"):
    """
    Saves simulation results in a structured, metadata-rich JSON format.
    Includes the numerical delta used for calculating the spectral index.
    """
    if not os.path.exists(output_dir):
        try:
            os.makedirs(output_dir)
        except OSError:
            pass # Handle race conditions

    # Generate unique run ID
    run_id = str(uuid.uuid4())[:8]
    timestamp = datetime.datetime.now().isoformat()
    
    # Organize data into a dictionary
    data = {
        "metadata": {
            "run_id": run_id,
            "timestamp": timestamp,
            "description": "Exact numerical calculation of ns and r from Mukhanov-Sasaki equations."
        },
        "model_parameters": {
            "name": model.name,
            "xi": getattr(model, 'xi_val', None),
            "lambda": getattr(model, 'lam', None),
            "phi0": getattr(model, 'phi0', None),
            "yi": getattr(model, 'yi', None),
            "S_parameter": model.S,
            "potential_v0": model.v0
        },
        "numerical_settings": {
            "delta_finite_difference": float(delta),
            "start_condition": "k = 100 * aH"
        },
        "background_info": {
            "total_efolds": float(N_total),
            "pivot_condition_N_star": 60.0
        },
        "pivot_scale": {
            "k_code_units": float(k_pivot_code),
            "exit_N_simulation": float(N_pivot)
        },
        "observables": {
            "n_s": float(ns),
            "r": float(r_val),
            "P_S_pivot": float(results_list[1][0]), # Middle value (at pivot)
            "P_T_pivot": float(results_list[1][1])
        },
        "spectrum_scan": [
            {
                "label": label,
                "k_code": float(k), 
                "P_S": float(Ps), 
                "P_T": float(Pt)
            } 
            for (label, k, (Ps, Pt)) in zip(["k_minus", "k_pivot", "k_plus"], ks_code_list, results_list)
        ]
    }
    
    # Construct descriptive filename
    safe_name = model.name.replace(' ', '_').replace('(', '').replace(')', '')
    xi_val = float(getattr(model, 'xi_val', 0) or 0)
    lam_val = float(getattr(model, 'lam', 0) or 0)
    phi0_val = float(getattr(model, 'phi0', 0) or 0)
    yi_val = float(getattr(model, 'yi', 0) or 0)
    
    filename = f"{safe_name}_xi{xi_val:.1f}_lambda{lam_val:.1e}_phi0_{phi0_val:.1f}_yi_{yi_val:.3f}_run_{run_id}.json"
    filepath = os.path.join(output_dir, filename)
    
    with open(filepath, 'w') as f:
        json.dump(data, f, indent=4)
        
    print(f"Results saved successfully to: {os.path.abspath(filepath)}")
    return filepath

# Execute with the delta we found to be stable
save_simulation_results(model, ns, r_val, delta, k_pivot_code, N_total, N_efolds[pivot_idx], results, ks_code)


Results saved successfully to: c:\Users\diego\OneDrive\Documentos\Universidad\Cosmologia\A-NumInflation\outputs\results\Higgs_Inflation_xi17000.0_lambda1.3e-01_phi0_6.0_yi_-0.100_run_86b342f5.json


'../outputs/results\\Higgs_Inflation_xi17000.0_lambda1.3e-01_phi0_6.0_yi_-0.100_run_86b342f5.json'